In [1]:
!git clone https://github.com/fujiry0/TrajICL.git
%cd TrajICL

Cloning into 'TrajICL'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 29 (delta 0), reused 29 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 10.18 MiB | 12.28 MiB/s, done.
/content/TrajICL


In [4]:
!sed -i 's/numpy==1.22.1/numpy>=1.22.1/g' requirements.txt

# 2. Try installing your requirements again
!pip install -r requirements.txt

  Using cached torch-2.3.1-cp312-cp312-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached torchvision-0.18.1-cp312-cp312-manylinux1_x86_64.whl.metadata (6.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 745.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Data of Motsynth

https://github.com/dvl-tum/motsynth-baselines/blob/main/docs/DATA_PREPARATION.md

In [3]:
MOTSYNTH_ROOT= 'data/motsynth'

# Download the MOT format (.txt) annotations
!wget -P $MOTSYNTH_ROOT https://motchallenge.net/data/MOTSynth_mot_annotations.zip

# Unzip to your dataset root
!unzip $MOTSYNTH_ROOT/MOTSynth_mot_annotations.zip -d $MOTSYNTH_ROOT

!rm $MOTSYNTH_ROOT/MOTSynth_mot_annotations.zip

--2026-04-14 16:24:51--  https://motchallenge.net/data/MOTSynth_mot_annotations.zip
Resolving motchallenge.net (motchallenge.net)... 131.159.19.34, 2a09:80c0:18::1034
Connecting to motchallenge.net (motchallenge.net)|131.159.19.34|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607592941 (579M) [application/zip]
Saving to: ‘data/motsynth/MOTSynth_mot_annotations.zip’

MOTSynth_mot_annota 100%[===================>] 579.45M  18.6MB/s    in 35s     

2026-04-14 16:25:26 (16.8 MB/s) - ‘data/motsynth/MOTSynth_mot_annotations.zip’ saved [607592941/607592941]

Archive:  data/motsynth/MOTSynth_mot_annotations.zip
   creating: data/motsynth/mot_annotations/
   creating: data/motsynth/mot_annotations/767/
   creating: data/motsynth/mot_annotations/767/gt/
  inflating: data/motsynth/mot_annotations/767/gt/gt.txt  
  inflating: data/motsynth/mot_annotations/767/seqinfo.ini  
   creating: data/motsynth/mot_annotations/143/
   creating: data/motsynth/mot_annotations/143/gt/

In [5]:
import os
import urllib.request
import zipfile

# Define the target data directory
data_dir = "data/motsynth"
annotations_zip_path = os.path.join(data_dir, "MOTSynth_mot_annotations.zip")

# Create directories if they do not exist
os.makedirs(data_dir, exist_ok=True)

# Step 1: Download the lightweight annotations
print("Downloading MOTSynth annotations (this may take a minute)...")
ann_url = "https://motchallenge.net/data/MOTSynth_mot_annotations.zip"
if not os.path.exists(annotations_zip_path):
    urllib.request.urlretrieve(ann_url, annotations_zip_path)
    print("Download complete.")
else:
    print("Annotations ZIP already exists. Skipping download.")

# Step 2: Extract the ZIP file
print("Extracting annotations...")
# This will extract folders into data/motsynth/mot_annotations/
with zipfile.ZipFile(annotations_zip_path, 'r') as zip_ref:
    zip_ref.extractall(data_dir)
print("Extraction complete.")

# Step 3: Download and process the train/val splits
print("Fetching official split files from motsynth-baselines...")
train_url = "https://raw.githubusercontent.com/dvl-tum/motsynth-baselines/main/tools/anns/splits/train.txt"
val_url = "https://raw.githubusercontent.com/dvl-tum/motsynth-baselines/main/tools/anns/splits/val.txt"

# The output paths that TrajICL's preprocess.py expects
train_txt_path = os.path.join(data_dir, "motsynth_train.txt")
val_txt_path = os.path.join(data_dir, "motsynth_val.txt")

def process_split(url, output_path):
    # Fetch the raw text file from GitHub
    response = urllib.request.urlopen(url)
    content = response.read().decode('utf-8')

    # Read numbers, pad with zeros to 3 digits, and format list
    processed_lines = []
    for line in content.strip().split('\n'):
        if line.strip():
            # Convert '42' to '042' to match the extracted folder names
            padded_id = f"{int(line.strip()):03d}"
            processed_lines.append(padded_id)

    # Write the properly formatted list to the local text file
    with open(output_path, 'w') as f:
        f.write('\n'.join(processed_lines))

# Execute the split processing
process_split(train_url, train_txt_path)
process_split(val_url, val_txt_path)
print("Success! motsynth_train.txt and motsynth_val.txt are ready.")

Annotations ZIP already exists. Skipping download.
Extracting annotations...
Extraction complete.
Fetching official split files from motsynth-baselines...
Success! motsynth_train.txt and motsynth_val.txt are ready.


# preprocess.py

* **Stage 1**: Data Parsing & Formatting (preprocess)First, it converts the raw data into a numerical format the neural network can understand.Feature Extraction: It reads the raw MOTSynth .txt files, calculates the center (x, y) coordinates from the bounding boxes, and downsamples the frame rate (e.g., from 30 fps to 2.5 fps).Sequence Chunking: It groups the continuous tracking data into fixed-length sequence chunks (e.g., 21 frames: 9 frames for history, 12 for the future).Pool/Valid Splitting: It splits the pedestrians into a "pool" set (trajectories that can be used as in-context examples) and a "valid" set (the trajectories the model will actually try to predict).Saving Tensors: It saves these numerical coordinates and their corresponding visibility masks to disk as PyTorch tensor files (.pt).

* **Stage 2**: Similarity Matrix Computation (sim_matrix)Because TrajICL relies on giving the model "similar" past trajectories as prompts, it needs to know mathematically which trajectories are similar to each other.It uses PyTorch (torch.cdist) to calculate the pairwise Euclidean distances between the coordinates of all trajectories.It also calculates the pairwise differences in their velocities.It normalizes and saves these massive similarity matrices to disk. (This is the step that requires a lot of RAM/VRAM).

* **Stage 3**: Building Trajectory Prompt Dictionaries (traj_sim)Finally, it ranks the similarities to finalize the prompt selection.It combines the distance and velocity matrices using a weighted formula.For every target trajectory in the validation set, it finds the top-$K$ (default is 16) most similar trajectories from the pool.It saves this mapping as a Python dictionary (.pickle file).

In [6]:
!python preprocess.py --name motsynth

===== Stage 1: preprocess (load_data -> pool/valid split -> save) =====
[Preprocess] name=motsynth, split=train
loaded train processed data !!!
motsynth train: 68870
average num people: 8.611238565413098
max num people: 65
min num people: 1
Trajectory Statistics:
  primary_mean: 0.8525966797347659
  primary_std: 78.52392614957016
  primary_min: -1803.0
  primary_max: 1816.0
  primary_velocity: -0.003205435361308746
  traj_max: 1816.0
  traj_min: -1852.5
********************
fold 0
pool data num: 52871
valid data num: 15999
valid data num filtered by min_prompt_num: 15875
********************
fold 1
pool data num: 53177
valid data num: 15693
valid data num filtered by min_prompt_num: 15595
********************
fold 2
pool data num: 54681
valid data num: 14189
valid data num filtered by min_prompt_num: 14098
********************
fold 3
pool data num: 56622
valid data num: 12248
valid data num filtered by min_prompt_num: 12162
********************
fold 4
pool data num: 59786
valid data nu

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!cp -r /content/TrajICL /content/drive/MyDrive/ECEN689_Multimodal_CV/Final

# Train

In [8]:
!pip install --upgrade setuptools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.0 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 64.0.3
    Uninstalling setuptools-64.0.3:
      Successfully uninstalled setuptools-64.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
%cd /content/TrajICL
!python train.py

/content/TrajICL
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 3
wandb: You chose "Don't visualize my results"
wandb: Tracking run with wandb version 0.17.8
wandb: W&B syncing is set to `offline` in this directory.  
wandb: Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
train: 66732 trajectories

val: 3152 trajectories
Error executing job with overrides: []
Traceback (most recent call last):
  File "/content/TrajICL/train.py", line 36, in main
    model = create_model(cfg)
            ^^^^^^^^^^^^^^^^^
  File "/content/TrajICL/model.py", line 141, in create_model
    model = Model(cfg).to(cfg["device"])
            ^^^^^^^^^^
  File "/content/TrajICL/model.py", line 18, in __init__
    self.dec_emb = DecoderEmbedding(
                   ^^^^^^^^^^^^^^^^^
  File "/content/TrajICL/embedding.py", line 133, in __init__
    self.emb = nn.Embedding(1000, d_model // 2, max_norm=True)